### Imports

In [1]:
from pathlib import Path
import pandas as pd

from src.config import RAW_PDF_DIR
from src.ingest.pdf_parser import (
    pdf_to_text,
    extract_report_week,
    find_all_district_rows,
    parse_wer_pdf,
    CANONICAL_RDHS
)

pdf_files = sorted(RAW_PDF_DIR.glob('*.pdf'))
print("PDFs: ", len(pdf_files))

PDFs:  1017


### Check a sample video

In [2]:
sample_files = []

sample_files.append(pdf_files[0])

for fraction in [0.2,0.4,0.6,0.8]:
    index = int(len(pdf_files) * fraction)
    sample_files.append(pdf_files[index])

sample_files.append(pdf_files[-1])
sample_files = list(dict.fromkeys(sample_files))

for path in sample_files:
    print(path.name)

wer_2007_w01.pdf
wer_2010_w47.pdf
wer_2014_w42.pdf
wer_2018_w37.pdf
wer_2022_w31.pdf
wer_2026_w27.pdf


### Inspect the first PDF

In [3]:
pdf_path = sample_files[4]

print(pdf_path)

text = pdf_to_text(pdf_path)

print(text[:100000])


/media/breezy/NewVolume/projects_int/dengue/data/raw/wer_pdfs/wer_2022_w31.pdf
                       WEEKLY EPIDEMIOLOGICAL REPORT
                                 A publication of the Epidemiology Unit
                          Ministry of Health, Nutrition & Indigenous Medicine
                                      231, de Saram Place, Colombo 01000, Sri Lanka
                          Tele: + 94 11 2695112, Fax: +94 11 2696583, E mail: epidunit@sltnet.lk
                                 Epidemiologist: +94 11 2681548, E mail: chepid@sltnet.lk
                                               Web: http://www.epid.gov.lk


                  Vol. 49 No. 31                                                                      30 th– 05th Aug 2022
 SRI LANKA 2022
                                           Can we eliminate Tuberculosis? Part II
                  This is the last article of series of two arti-        If yes, how?
                  cles.                                        

### Test report date extraction

In [4]:
rows = find_all_district_rows(text)

print(
    "Districts found:",
    len(rows)
)

for district in CANONICAL_RDHS:

    values = rows.get(district)

    if values is None:
        print(
            f"{district:20} MISSING"
        )
    else:
        print(
            f"{district:20} "
            f"numbers={len(values):2d} "
            f"first_values={values[:8]}"
        )

Districts found: 26
Colombo              numbers=26 first_values=[38, 8215, 0, 4, 0, 3, 0, 0]
Gampaha              numbers=26 first_values=[17, 4803, 0, 5, 0, 1, 0, 0]
Kalutara             numbers=26 first_values=[17, 2560, 1, 12, 0, 1, 0, 1]
Kandy                numbers=26 first_values=[23, 2904, 3, 15, 0, 0, 0, 2]
Matale               numbers=26 first_values=[36, 669, 0, 2, 0, 0, 0, 0]
Nuwara Eliya         numbers=26 first_values=[9, 152, 1, 15, 0, 0, 0, 2]
Galle                numbers=26 first_values=[15, 2582, 0, 7, 0, 0, 0, 0]
Matara               numbers=26 first_values=[59, 1063, 1, 12, 1, 2, 0, 0]
Hambantota           numbers=26 first_values=[76, 972, 0, 24, 0, 0, 0, 0]
Jaffna               numbers=26 first_values=[46, 2324, 4, 41, 0, 2, 1, 57]
Kilinochchi          numbers=26 first_values=[1, 93, 0, 6, 0, 0, 0, 1]
Mannar               numbers=26 first_values=[1, 174, 1, 2, 0, 0, 0, 0]
Vavuniya             numbers=26 first_values=[0, 66, 1, 3, 0, 1, 0, 2]
Mullaitivu           nu

### import

In [5]:
from pathlib import Path

import pandas as pd

from src.config import (
    DENGUE_BRONZE_PATH,
    DENGUE_SILVER_PATH,
)

from src.cleaning.dengue_cleaner import (
    clean_dengue_panel,
    create_complete_panel,
)

from src.cleaning.validation import (
    validate_panel,
    print_validation_report,
)

### read the dataset

In [6]:
raw = pd.read_parquet(
    DENGUE_BRONZE_PATH
)

raw.head()

,district,source_file,week_ending,epi_week,year,dengue_fever_this_week,dengue_fever_cumulative,dysentery_this_week,dysentery_cumulative,encephalitis_this_week,...,meningitis_this_week,meningitis_cumulative,leishmaniasis_this_week,leishmaniasis_cumulative,timeliness_pct,completeness_pct,tuberculosis_this_week,tuberculosis_cumulative,leprosy_this_week,leprosy_cumulative
0,Colombo,wer_2007_w01.pdf,2006-12-29,52,2006,71,3419,2,356,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Gampaha,wer_2007_w01.pdf,2006-12-29,52,2006,12,1774,0,331,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Kalutara,wer_2007_w01.pdf,2006-12-29,52,2006,12,977,6,506,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Kandy,wer_2007_w01.pdf,2006-12-29,52,2006,20,1509,5,452,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Matale,wer_2007_w01.pdf,2006-12-29,52,2006,4,389,0,344,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [7]:
print(
    raw.shape
)

print(
    raw.columns.tolist()
)

(26441, 36)
['district', 'source_file', 'week_ending', 'epi_week', 'year', 'dengue_fever_this_week', 'dengue_fever_cumulative', 'dysentery_this_week', 'dysentery_cumulative', 'encephalitis_this_week', 'encephalitis_cumulative', 'enteric_fever_this_week', 'enteric_fever_cumulative', 'food_poisoning_this_week', 'food_poisoning_cumulative', 'leptospirosis_this_week', 'leptospirosis_cumulative', 'typhus_this_week', 'typhus_cumulative', 'viral_hepatitis_this_week', 'viral_hepatitis_cumulative', 'returns_pct', 'human_rabies_this_week', 'human_rabies_cumulative', 'chickenpox_this_week', 'chickenpox_cumulative', 'meningitis_this_week', 'meningitis_cumulative', 'leishmaniasis_this_week', 'leishmaniasis_cumulative', 'timeliness_pct', 'completeness_pct', 'tuberculosis_this_week', 'tuberculosis_cumulative', 'leprosy_this_week', 'leprosy_cumulative']


### clean the dengue panel

In [8]:
clean = clean_dengue_panel(
    raw
)

clean.head()

[clean] drop unparseable year/week: 26441 → 26441 rows (Δ 0)
[clean] filter weeks 1-54: 26441 → 26441 rows (Δ 0)
[clean] filter canonical districts: 26441 → 26441 rows (Δ 0)
[clean] drop null week_start: 26441 → 26441 rows (Δ 0)


[WARNING] Duplicate district/week observations detected (104 rows):
    district  year  week week_start      source_file  cases
      Ampara  2016    53 2017-01-02 wer_2016_w53.pdf      0
      Ampara  2017     1 2017-01-02 wer_2017_w01.pdf      5
      Ampara  2021    53 2022-01-03 wer_2021_w53.pdf      1
      Ampara  2022     1 2022-01-03 wer_2022_w01.pdf      2
Anuradhapura  2016    53 2017-01-02 wer_2016_w53.pdf      7
Anuradhapura  2017     1 2017-01-02 wer_2017_w01.pdf     16
Anuradhapura  2021    53 2022-01-03 wer_2021_w53.pdf     13
Anuradhapura  2022     1 2022-01-03 wer_2022_w01.pdf     13
     Badulla  2016    53 2017-01-02 wer_2016_w53.pdf     18
     Badulla  2017     1 2017-01-02 wer_2017_w01.pdf     40
     Badulla  2021    53 2022-01-03 wer_2021_w53.pdf     78
     Badulla  2022     1 2022-01-03 wer_2022_w01.pdf     78
  Batticaloa  2016    53 2017-01-02 wer_2016_w53.pdf     17
  Batticaloa  2017     1 2017-01-02 wer_2017_w01.pdf     43
  Batticaloa  2021    53 2022-01

[WARNING] Duplicate district/week observations detected (104 rows):
    district  year  week week_start      source_file  cases
      Ampara  2016    53 2017-01-02 wer_2016_w53.pdf      0
      Ampara  2017     1 2017-01-02 wer_2017_w01.pdf      5
      Ampara  2021    53 2022-01-03 wer_2021_w53.pdf      1
      Ampara  2022     1 2022-01-03 wer_2022_w01.pdf      2
Anuradhapura  2016    53 2017-01-02 wer_2016_w53.pdf      7
Anuradhapura  2017     1 2017-01-02 wer_2017_w01.pdf     16
Anuradhapura  2021    53 2022-01-03 wer_2021_w53.pdf     13
Anuradhapura  2022     1 2022-01-03 wer_2022_w01.pdf     13
     Badulla  2016    53 2017-01-02 wer_2016_w53.pdf     18
     Badulla  2017     1 2017-01-02 wer_2017_w01.pdf     40
     Badulla  2021    53 2022-01-03 wer_2021_w53.pdf     78
     Badulla  2022     1 2022-01-03 wer_2022_w01.pdf     78
  Batticaloa  2016    53 2017-01-02 wer_2016_w53.pdf     17
  Batticaloa  2017     1 2017-01-02 wer_2017_w01.pdf     43
  Batticaloa  2021    53 2022-01

,district,source_file,week_ending,epi_week,year,dengue_fever_this_week,dengue_fever_cumulative,dysentery_this_week,dysentery_cumulative,encephalitis_this_week,...,leishmaniasis_cumulative,timeliness_pct,completeness_pct,tuberculosis_this_week,tuberculosis_cumulative,leprosy_this_week,leprosy_cumulative,cases,week,week_start
0,Ampara,wer_2007_w01.pdf,2006-12-29,52,2007,0,33,2,263,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,1,2007-01-01
1,Ampara,wer_2007_w02.pdf,2007-01-05,1,2007,0,0,2,2,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,2,2007-01-08
2,Ampara,wer_2007_w03.pdf,2007-01-12,2,2007,0,0,1,4,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,3,2007-01-15
3,Ampara,wer_2007_w04.pdf,2007-01-19,3,2007,0,0,2,14,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,4,2007-01-22
4,Ampara,wer_2007_w05.pdf,2007-01-26,4,2007,0,0,0,16,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,5,2007-01-29


In [9]:
clean.shape

(26389, 39)